In [ ]:
# %%
#Import top level libraries, including the deepvelo package
import numpy as np
import scvelo as scv
import torch

from deepvelo.utils import velocity, update_dict
from deepvelo.utils.preprocess import autoset_coeff_s
from deepvelo.utils.plot import statplot, compare_plot
from deepvelo import train, Constants

# fix random seeds for reproducibility
""" SEED = 123
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
np.random.seed(SEED) """

# set options for for visualization and verbosity
scv.settings.verbosity = 3  # show errors(0), warnings(1), info(2), hints(3)
scv.settings.set_figure_params(
    "scvelo", transparent=False
)  # for beautified visualization

%load_ext autoreload
%autoreload 2


In [ ]:
SEED = 123

import random
import numpy as np
import torch
import scanpy as sc

# python
random.seed(SEED)

# numpy
np.random.seed(SEED)

# pytorch
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# cuda deterministic
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# scanpy / scvelo
sc.settings.seed = SEED

In [ ]:
import scanpy as sc
import scvelo as scv
adata = sc.read_h5ad(
    "../../processed_conbine_data.h5ad"
)

print(adata)
print("layers:", adata.layers.keys())
print("raw:", adata.raw)
#print(adata.obs)
adata.obs["condition"].unique()

In [ ]:
scv.pp.filter_and_normalize(adata, min_shared_counts=20, n_top_genes=2000)
scv.pp.moments(adata, n_neighbors=30, n_pcs=30)

In [ ]:
adata.obs['celltype.anno'].value_counts()

In [ ]:
configs = {
    "name": "DeepVelo", # name of the experiment
    "loss": {"args": {"coeff_s": autoset_coeff_s(adata)}} # Automatic setting of the spliced correlation objective
}
configs = update_dict(Constants.default_configs, configs)


In [ ]:
# initial velocity
velocity(adata, mask_zero=False)
trainer = train(adata, configs)

In [ ]:
adata.write("../data/deepvelo_output_2.h5ad")

In [ ]:
adata.obs['root_cells'] = adata.obs['celltype.anno'] == 'Stem Cell Niche'

In [ ]:
adata.obsm.keys()

In [ ]:
# 设置图片保存目录
scv.settings.figdir = "other_photo/deepvelo"
import matplotlib.pyplot as plt

In [ ]:
import numpy as np

v = adata.layers.get('velocity')
print(type(v), v.shape)
print(np.isnan(v).sum())  # 是否有 NaN

In [ ]:
import scvelo as scv

# 假设 velocity 是基于 2000 HVG 的 X_pca
scv.pp.pca(adata, n_comps=50)          # 确保有 PCA
scv.pp.neighbors(adata, n_neighbors=30, n_pcs=50)  # 与 PCA 对应

In [ ]:
adata.layers['velocity'] = np.nan_to_num(adata.layers['velocity'])

In [ ]:
import scanpy as sc
import scvelo as scv
adata = sc.read_h5ad(
    "../data/deepvelo_output.h5ad"
)

print(adata)
print("layers:", adata.layers.keys())
print("raw:", adata.raw)
#print(adata.obs)
adata.obs["condition"].unique()

In [ ]:
print(adata.n_obs, adata.n_vars)
print(adata.layers['velocity'].shape)

# 检查 neighbors 图
print(adata.obsp['distances'].shape)
print(adata.obsp['connectivities'].shape)

In [ ]:
""" scv.tl.velocity_graph(adata, n_jobs=8) """

In [ ]:
import scvelo as scv
import scanpy as sc

# 1 过滤细胞
#adata = adata[adata.obs['celltype.anno'] != 'Trichoblast'].copy()

""" # 2 重新 neighbors
sc.pp.neighbors(adata)

# 3 重新 moments
scv.pp.moments(adata) """

# 4 重新计算 velocity
scv.tl.velocity(adata)

# 5 重新 velocity graph
#scv.tl.velocity_graph(adata)

# 6 设置 root
adata.obs['root_cells'] = adata.obs['celltype.anno'] == 'Stem Cell Niche'

# 7 重新 latent time
#scv.tl.latent_time(adata)

In [ ]:
import scvelo as scv
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6,6))

scv.pl.velocity_embedding_stream(
    adata,
    basis="umap",
    color="celltype.anno",
    legend_loc=None,
    ax=ax,
    show=False
)

ax.set_title("DeepVelo", fontsize=16)
fig.savefig("other_photo/deepvelo/deepvelo_velocity_stream.png", dpi=600, bbox_inches="tight")
plt.close(fig)

In [ ]:
""" scv.tl.velocity_pseudotime(adata, vkey="velocity")

scv.pl.scatter(
    adata,
    basis="umap",
    color="velocity_pseudotime",
    cmap="plasma"
) """
import scvelo as scv
import matplotlib.pyplot as plt
import os

# 计算 velocity pseudotime
scv.tl.velocity_pseudotime(adata, vkey="velocity")

# 创建保存目录
os.makedirs("other_photo/deepvelo", exist_ok=True)

# 创建 figure
fig, ax = plt.subplots(figsize=(6,6))

# 画图
scv.pl.scatter(
    adata,
    basis="umap",
    color="velocity_pseudotime",
    cmap="plasma",
    ax=ax,
    show=False
)

# 添加标题
ax.set_title("DeepVelo Velocity Pseudotime", fontsize=16)

# 保存图片
fig.savefig(
    "other_photo/deepvelo/deepvelo_velocity_pseudotime.png",
    dpi=600,
    bbox_inches="tight"
)

plt.close(fig)

In [ ]:
""" import scanpy as sc
fig, ax = plt.subplots(figsize=(6,6))
scv.pl.umap(
    adata,
    color=["celltype.anno", "condition"],
    wspace=0.4
)

ax.set_title("DeepVelo", fontsize=16)
fig.savefig("other_photo/deepvelo/condition.png", dpi=600, bbox_inches="tight")
plt.close(fig) """

import scvelo as scv
import matplotlib.pyplot as plt
import os

# 创建目录
os.makedirs("other_photo/deepvelo", exist_ok=True)

# 画图
scv.pl.umap(
    adata,
    color=["celltype.anno", "condition"],
    wspace=0.4,
    show=False
)

# 获取当前figure
fig = plt.gcf()

# 加标题
fig.suptitle("DeepVelo", fontsize=16)

# 保存
fig.savefig(
    "other_photo/deepvelo/deepvelo_condition.png",
    dpi=600,
    bbox_inches="tight"
)

plt.close(fig)

In [ ]:
""" scv.tl.latent_time(adata)
fig, ax = plt.subplots(figsize=(6,6))
scv.pl.scatter(
    adata,
    color="latent_time",
    cmap="viridis"
) """

import scvelo as scv
import matplotlib.pyplot as plt
import os

# 计算 latent time
scv.tl.latent_time(adata)

# 创建目录
os.makedirs("other_photo/deepvelo", exist_ok=True)

# 创建图
fig, ax = plt.subplots(figsize=(6,6))

# 画图
scv.pl.scatter(
    adata,
    basis="umap",
    color="latent_time",
    cmap="viridis",
    size=30,
    ax=ax,
    show=False
)

# 标题
ax.set_title("DeepVelo Latent Time", fontsize=16)

# 保存
fig.savefig(
    "other_photo/deepvelo/deepvelo_latent_time.png",
    dpi=600,
    bbox_inches="tight"
)

plt.close(fig)

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# expression matrix
X = adata.layers["spliced"]
if not isinstance(X, np.ndarray):
    X = X.toarray()

# velocity matrix
V = adata.layers["velocity"]
if not isinstance(V, np.ndarray):
    V = V.toarray()

# neighbor graph
connectivities = adata.obsp["connectivities"]

scores = []

for i in range(adata.n_obs):

    neighbors = connectivities[i].indices

    if len(neighbors) == 0:
        scores.append(0)
        continue

    vi = V[i].reshape(1, -1)

    sims = []

    for j in neighbors:
        delta = (X[j] - X[i]).reshape(1, -1)

        sim = cosine_similarity(vi, delta)[0][0]
        sims.append(sim)

    scores.append(np.mean(sims))

adata.obs["velocity_consistency"] = scores

In [ ]:
""" import scvelo as scv

scv.pl.scatter(
    adata,
    basis="umap",
    color="velocity_consistency", 
    cmap="coolwarm"
) """

import scvelo as scv
import matplotlib.pyplot as plt
import os

# 创建目录
os.makedirs("other_photo/deepvelo", exist_ok=True)

# 创建图
fig, ax = plt.subplots(figsize=(6,6))

# 画 velocity consistency
scv.pl.scatter(
    adata,
    basis="umap",
    color="velocity_consistency",
    cmap="coolwarm",
    #zise=30,
    ax=ax,
    show=False
)

# 添加标题
ax.set_title("Velocity Consistency (DeepVelo)", fontsize=16)

# 保存图片
fig.savefig(
    "other_photo/deepvelo/deepvelo_velocity_consistency.png",
    dpi=600,
    bbox_inches="tight"
)

plt.close(fig)

In [ ]:
np.mean(adata.obs["velocity_consistency"])

In [ ]:
import scvelo as scv

scv.tl.velocity_confidence(adata)

In [ ]:
import scvelo as scv
import matplotlib.pyplot as plt
import os

os.makedirs("other_photo/deepvelo", exist_ok=True)

fig, ax = plt.subplots(figsize=(6,6))

scv.pl.scatter(
    adata,
    basis="umap",
    color="velocity_confidence",
    cmap="coolwarm",
    ax=ax,
    show=False
)

ax.set_title("Velocity Confidence (DeepVelo)", fontsize=16)

fig.savefig(
    "other_photo/deepvelo/deepvelo_velocity_confidence.png",
    dpi=600,
    bbox_inches="tight"
)

plt.close(fig)

In [ ]:
import scvelo as scv

scv.tl.velocity_confidence(adata)

In [ ]:
import scvelo as scv
import pandas as pd
import numpy as np

def velocity_benchmark(adata, model_name):

    results = {}

    # 1 velocity confidence
    if "velocity_confidence" not in adata.obs:
        scv.tl.velocity_confidence(adata)

    conf = adata.obs["velocity_confidence"]

    results["model"] = model_name
    results["confidence_mean"] = np.mean(conf)
    results["confidence_std"] = np.std(conf)

    # 2 velocity consistency
    if "velocity_consistency" not in adata.obs:
        scv.tl.velocity_confidence(adata)

    cons = adata.obs["velocity_consistency"]

    results["consistency_mean"] = np.mean(cons)
    results["consistency_std"] = np.std(cons)

    # 3 velocity pseudotime
    if "velocity_pseudotime" not in adata.obs:
        scv.tl.velocity_pseudotime(adata)

    pt = adata.obs["velocity_pseudotime"]

    results["pseudotime_mean"] = np.mean(pt)
    results["pseudotime_std"] = np.std(pt)

    # 4 latent time
    if "latent_time" not in adata.obs:
        scv.tl.latent_time(adata)

    lt = adata.obs["latent_time"]

    results["latent_time_mean"] = np.mean(lt)
    results["latent_time_std"] = np.std(lt)

    # 5 velocity magnitude
    v = adata.layers["velocity"]
    mag = np.linalg.norm(v, axis=1)

    results["velocity_mag_mean"] = np.mean(mag)
    results["velocity_mag_std"] = np.std(mag)

    return results

In [ ]:
results = []
results.append(velocity_benchmark(adata, "deepvelo"))

df = pd.DataFrame(results)

print(df)

In [ ]:
results = []

results.append(velocity_benchmark(adata, "scvelo"))
results.append(velocity_benchmark(adata, "deepvelo"))
results.append(velocity_benchmark(adata, "transvelo"))

df = pd.DataFrame(results)

print(df)